In [2]:
import h5py
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import yaml
from IPython.display import Audio
from diffusers import DiffusionPipeline
from scipy import signal

import utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
class EEGProjector(nn.Module):

    def __init__(self, Fy, Sy, Fz, Sz):
        super(EEGProjector, self).__init__()
        self.Fy = Fy  # eeg channels
        self.Sy = Sy  # time steps
        self.Fz = Fz  # freq
        self.Sz = Sz  # time

        self.proj_layers = nn.Sequential(
            nn.Conv1d(Fy, 256, kernel_size=5, stride=5),
            nn.ReLU(),
            nn.Conv1d(256, 512, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv1d(512, 1024, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv1d(1024, 2048, kernel_size=2, stride=2),
            nn.ReLU(),
        )

        self.fc = nn.Linear(2048 * 6, Fz * Sz)

    def forward(self, eeg):
        eeg_projected = self.proj_layers(eeg)
        batch_size, reduced_Fy, reduced_Sy = eeg_projected.shape
        eeg_projected = eeg_projected.view(batch_size, -1)
        eeg_projected = self.fc(eeg_projected)
        eeg_projected = eeg_projected.view(batch_size, 1, self.Fz, self.Sz)
        return eeg_projected

In [16]:
pipeline = DiffusionPipeline.from_pretrained("cvssp/audioldm2-music")
pipeline = pipeline.to(device)

prompt_embeds, attention_mask, generated_prompt_embeds = pipeline.encode_prompt(
    prompt='Pop music',
    device=device,
    num_waveforms_per_prompt=1,
    do_classifier_free_guidance=False
)


class ControlNet(nn.Module):
    def __init__(self, base_model, Fy, Sy, Fz, Sz):
        super(ControlNet, self).__init__()
        self.base_model = base_model
        self.unet = base_model.unet
        self.projector = EEGProjector(Fy, Sy, Fz, Sz)
        self.zero_conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        nn.init.zeros_(self.zero_conv.weight)
        nn.init.zeros_(self.zero_conv.bias)

    def forward(self, z, eeg, t):
        eeg_projected = self.projector(eeg)
        condition = self.zero_conv(eeg_projected) + self.zero_conv(z)
        condition = condition.squeeze(1)
        #print(condition.shape)
        pred = self.unet(self.zero_conv(z), t, encoder_hidden_states=condition.mean(dim=1))
        return pred

Loading pipeline components...:   0%|          | 0/11 [00:00<?, ?it/s]

In [5]:
path = 'data'
data = h5py.File(os.path.join(path, 'madeeg_preprocessed.hdf5'), 'r')
stream = open(os.path.join(path, 'madeeg_preprocessed.yaml'), 'r')
metadata = yaml.load(stream, Loader=yaml.FullLoader)

fs = 256
num_sec = 1

eeg_X = []
spec_y = []
audio_z = []
subjects = list(data.keys())
for sbj in subjects[:1]:

    stimuli = list(metadata[sbj].keys())
    for stim in stimuli[:2]:

        response = data[sbj][stim]['response']
        interval_length = response.shape[1] // 4
        mean_eeg = torch.zeros_like(torch.empty(20, interval_length))

        for i in range(4):
            start_index = i * interval_length
            end_index = (i + 1) * interval_length
            interval = response[:, start_index:end_index]
            mean_eeg += interval
        mean_eeg /= 4

        seg = 0
        while (seg + num_sec) * fs < mean_eeg.shape[1]:
            eeg_tensor = mean_eeg[:, seg * fs:(seg + num_sec) * fs]

            stimulus = data[sbj][stim]['stimulus']
            sfreq = metadata[sbj][stim]['wav_info']['sfreq']

            # placed here due to mel-spec requiring sfreq
            pipeline.feature_extractor.sampling_rate = sfreq

            # audio to mono
            ch1 = stimulus[0, :]
            ch2 = stimulus[1, :]
            mix = ch1 + ch2
            mix = mix[seg * sfreq:(seg + num_sec) * sfreq]
            transform = T.MelSpectrogram()

            # generating spectrogram (ideally mel-spec)
            # spec_tensor = torch.tensor(np.array(pipeline.feature_extractor(mix, sampling_rate=sfreq)['input_features']))
            spec_tensor = transform(torch.tensor(mix).float().unsqueeze(0)).to(device)
            eeg_X.append(eeg_tensor)
            spec_y.append(spec_tensor)

            # computing noisy latent audio
            with torch.no_grad():
                latent_representation = pipeline.vae.tiled_encode(spec_tensor.unsqueeze(0)).latent_dist.sample()
            noise = torch.randn_like(latent_representation)
            z = latent_representation + noise
            audio_z.append(z)

            seg += 1
    print(f'subject {sbj} data gathered')

C:\Users\jlama\anaconda3\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


subject 0001 data gathered


In [6]:
print(f'{len(eeg_X)} samples gathered')

n_channels, n_times = eeg_X[0].shape
n_freqs, n_bins = spec_y[0][0].shape
Fz, Sz, Dz = audio_z[0][0].shape

print(f'{n_channels} channels per eeg recording')
print(f'{n_times} time steps per eeg recording')
print(f'{n_freqs} frequencies per spectrogram image')
print(f'{n_bins} time steps per spectrogram image')
print(f'{Fz} latent channels dimension')
print(f'{Sz} sequence y dimension')
print(f'{Dz} latent z dimension')

14 samples gathered
20 channels per eeg recording
256 time steps per eeg recording
128 frequencies per spectrogram image
221 time steps per spectrogram image
8 latent channels dimension
32 sequence y dimension
55 latent z dimension


In [17]:
controlnet = ControlNet(pipeline,
                        n_channels,
                        n_times,
                        n_freqs,
                        n_bins).to(device)
controlnet.train()
optimizer = optim.Adam(controlnet.parameters(), lr=1e-4)
criterion = nn.MSELoss()

num_epochs = 100
batch_size = 1

for epoch in range(num_epochs):
    controlnet.train()
    running_loss = 0.0

    combined = list(zip(eeg_X, spec_y))
    random.shuffle(combined)
    eeg_X_shuffled, spec_y_shuffled = zip(*combined)

    for i in range(0, len(eeg_X_shuffled), batch_size):
        batch_eeg = torch.stack(eeg_X_shuffled[i:i + batch_size]).float().to(device)
        batch_z = torch.stack(spec_y_shuffled[i:i + batch_size]).float().to(device)
        batch_z = batch_z.squeeze(1)
        t = torch.randint(0, pipeline.scheduler.num_train_timesteps, (batch_size,))
        optimizer.zero_grad()

        projection = controlnet(batch_z, batch_eeg, t)
        #output = decoder(projection)
        loss = criterion(projection, batch_z)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / (len(eeg_X_shuffled) / batch_size)
    print(f'Epoch {epoch + 1}, Loss: {avg_loss:.6f}')

C:\Users\jlama\anaconda3\Lib\site-packages\diffusers\configuration_utils.py:140: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDIMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDIMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x221 and 768x256)